# Restaurant Recommender: from data to agents

Select the project's Python 3.12 `.venv` kernel. Start Ollama before the LLM cells.
The processed dataset and recipe images must be present (see `data/README.md`).
Cells do not run automatically; agent execution can take around a minute.
Optional: start `make phoenix` in a terminal before running the agents.

In [ ]:
from pathlib import Path
import os

# Jupyter may start in notebooks/ instead of the project root.
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
assert Path('restaurant_ai').is_dir(), 'Open this notebook from the project folder'
from restaurant_ai import config
print('Text:', config.TEXT_LLM_MODEL, '| Vision:', config.VISION_LLM_MODEL)

## 1. Inspect structured data
A restaurant has an ID, location, cuisine, price, and other structured attributes.
Pydantic validates newly extracted records. Existing course data is kept unchanged.

In [ ]:
from restaurant_ai.data_pipeline.store import load_restaurants, format_restaurant_card
from evals.metrics import data_quality_report

restaurants = load_restaurants()
print(format_restaurant_card(restaurants[0]))
data_quality_report(restaurants)

## 2. Retrieve with text and images
Run `make index` once before this cell. Restaurant search uses MiniLM.
Recipe search combines normalized MiniLM and CLIP scores by recipe ID.

In [ ]:
from restaurant_ai.rag.retriever import search_restaurants, search_recipes

print(search_restaurants('cozy Japanese noodles', k=3, max_price=2))
search_recipes('spicy noodles', k=3)

## 3. Discover and call a real MCP tool
The client starts a Python MCP server using stdio. The context manager closes it afterward.
Jupyter supports top-level `await`, so do not use `asyncio.run` here.

In [ ]:
from restaurant_ai.agents.mcp_client import open_mcp_tools, call_tool_json

async with open_mcp_tools() as tools:
    print('Discovered tools:', list(tools))
    details = await call_tool_json(tools, 'get_restaurant_details',
                                   {'restaurant_name': restaurants[0]['name']})
details

## 4. Inspect the graph and run all five agents
Profile -> retrieval -> parallel cuisine/nutrition -> recommendations.
Parallel graph nodes do not guarantee concurrent model inference on a memory-limited machine.

In [ ]:
from restaurant_ai.agents.graph import workflow, run_recommendation
from restaurant_ai.agents.formatting import format_recommendations, format_agent_trace
from IPython.display import Markdown, display

print(workflow.get_graph().draw_mermaid())
state = await run_recommendation('Spicy Szechuan food and a recipe to cook at home')
display(Markdown(format_recommendations(state)))
display(Markdown(format_agent_trace(state)))

## 5. Measure, don't just eyeball
Completeness checks counts; grounding checks names against retrieved candidates.
Neither proves every explanation is factually correct. Run `make eval-fast` and
`python -m evals.run_evals --limit 1` in the terminal for persistent reports.

In [ ]:
from evals.metrics import completeness, groundedness, llm_tool_use

final = state['final_recommendations']
{'completeness': completeness(final),
 'name_grounding': groundedness(final),
 'model_selected_tools': llm_tool_use(state['tool_calls'])}